# Hoarding Detector — Training Notebook (AMD ROCm)
### TCS x AMD Hackathon · trains a YOLOv8 model that finds the boundary hoarding so the ad is placed there

**Why this exists:** the main pipeline currently locates the boundary board with
OpenCV colour/geometry rules, which break on camera angles, lighting and close-ups.
This notebook trains a proper **object detector** for the hoarding. The output
`hoarding_best.pt` plugs into the main pipeline and replaces the heuristic.

**You don't need any images.** There is no public "cricket boundary LED-board"
dataset, so we generate **synthetic frames with domain randomization** where the
hoarding location is known exactly, and auto-write YOLO labels. A clearly marked
**Option B** section shows how to add a few hundred real frames later to close the
synthetic→real gap (recommended before the final demo).

Pipeline: synthesize labelled frames → train YOLOv8n on the AMD GPU → evaluate →
export `hoarding_best.pt` → drop the integration snippet into the main notebook.


## 1 · Install + GPU check

In [ ]:
import shutil, subprocess, sys
def sh(c): return subprocess.run(c, shell=True, capture_output=True, text=True)
%pip install -q ultralytics opencv-python-headless
import torch
print("torch:", torch.__version__, "| ROCm/HIP:", getattr(torch.version,"hip",None))
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
DEVICE = 0 if torch.cuda.is_available() else "cpu"
print("Training device ->", DEVICE)


## 2 · Config

In [ ]:
from pathlib import Path
DS        = Path("./hoarding_ds")     # dataset output
IMG_W, IMG_H = 640, 384
N_TRAIN   = 800
N_VAL     = 200
NEG_FRAC  = 0.15                       # fraction of frames with NO hoarding (close-ups)
EPOCHS    = 60
BATCH     = 16
MODEL_OUT = "hoarding_best.pt"
for sub in ["images/train","images/val","labels/train","labels/val"]:
    (DS/sub).mkdir(parents=True, exist_ok=True)
print("Config ready ->", DS.resolve())


## 3 · Synthetic frame generator (with domain randomization)
Each call returns an image plus the hoarding bounding box (or `None` for a negative
close-up). Randomized: shot type (wide↔medium), crowd colour/height, board slope,
partial-width boards, ad-panel colours, pitch/players/ball distractors, brightness.
The variety is what lets the model generalize to real footage.

In [ ]:
import numpy as np, cv2

def synth_frame(rng, positive=True, w=IMG_W, h=IMG_H):
    img = np.zeros((h, w, 3), np.uint8)
    if not positive:
        # close-up / empty field: green everywhere, no boundary board in view
        g = np.array([rng.integers(30,55), rng.integers(95,140), rng.integers(30,55)])
        for y in range(h):
            t=y/h; img[y]=(g*(1-0.25*t)).astype(np.uint8)
        if rng.random()<0.7:
            px=int(rng.uniform(0.3,0.7)*w); pw=int(rng.uniform(0.06,0.12)*w)
            cv2.rectangle(img,(px,0),(px+pw,h),(120,165,120),-1)
        for _ in range(rng.integers(0,4)):
            cx,cy=int(rng.uniform(0.1,0.9)*w),int(rng.uniform(0.2*h,h-10))
            cv2.rectangle(img,(cx,cy),(cx+rng.integers(8,18),cy+rng.integers(18,34)),
                          (int(rng.integers(0,255)),)*3,-1)
        img=np.clip(img.astype(np.float32)*rng.uniform(0.7,1.2),0,255).astype(np.uint8)
        return img, None

    crowd_h = int(rng.uniform(0.08, 0.30) * h)
    board_h = int(rng.uniform(0.035, 0.075) * h)
    slopeL  = int(rng.uniform(-0.02, 0.02) * h)
    slopeR  = int(rng.uniform(-0.02, 0.02) * h)
    ft = crowd_h + board_h
    cb = np.array([rng.integers(70,120), rng.integers(70,115), rng.integers(90,150)])
    img[:crowd_h] = np.clip(cb + rng.integers(-30,30,(crowd_h,w,3)),0,255).astype(np.uint8)
    g_top = np.array([rng.integers(30,55), rng.integers(95,140), rng.integers(30,55)])
    g_bot = g_top * rng.uniform(0.7,0.95)
    for y in range(ft, h):
        t=(y-ft)/max(1,h-ft); img[y]=(g_top*(1-t)+g_bot*t).astype(np.uint8)

    bx0=int(rng.uniform(0.0,0.15)*w); bx1=int(rng.uniform(0.85,1.0)*w)
    base=(int(rng.integers(30,70)),)*3
    topL,topR,botL,botR = crowd_h+slopeL, crowd_h+slopeR, ft+slopeL, ft+slopeR
    cv2.fillPoly(img,[np.array([[bx0,topL],[bx1,topR],[bx1,botR],[bx0,botL]],np.int32)],base)
    palette=[(40,40,210),(50,200,230),(210,210,210),(60,60,200),(200,140,40),(40,160,80)]
    x=bx0+5
    while x<bx1-30:
        pw=int(rng.uniform(0.06,0.14)*w)
        yy=int(crowd_h+(slopeL+(slopeR-slopeL)*(x-bx0)/max(1,bx1-bx0)))
        cv2.rectangle(img,(x,yy+3),(min(x+pw,bx1-3),yy+board_h-3),
                      palette[rng.integers(0,len(palette))],-1)
        x+=pw+int(rng.uniform(0.01,0.05)*w)

    if rng.random()<0.7:
        px=int(rng.uniform(0.3,0.7)*w); pw=int(rng.uniform(0.06,0.12)*w)
        cv2.rectangle(img,(px,ft+10),(px+pw,h-10),(120,165,120),-1)
    for _ in range(rng.integers(0,4)):
        cx,cy=int(rng.uniform(0.1,0.9)*w),int(rng.uniform(ft+10,h-10))
        cv2.rectangle(img,(cx,cy),(cx+rng.integers(8,18),cy+rng.integers(18,34)),
                      (int(rng.integers(0,255)),)*3,-1)
    img=np.clip(img.astype(np.float32)*rng.uniform(0.7,1.2),0,255).astype(np.uint8)

    y_top,y_bot=min(topL,topR),max(botL,botR)
    box=(0,((bx0+bx1)/2)/w,((y_top+y_bot)/2)/h,(bx1-bx0)/w,(y_bot-y_top)/h)
    return img, box

print("synth_frame() ready")


## 4 · Generate the labelled dataset (YOLO format) + data.yaml

In [ ]:
import numpy as np, cv2

def build_split(split, n, seed):
    rng=np.random.default_rng(seed)
    for i in range(n):
        positive = rng.random() > NEG_FRAC
        img, box = synth_frame(rng, positive=positive)
        stem=f"{split}_{i:05d}"
        cv2.imwrite(str(DS/f"images/{split}/{stem}.jpg"), img)
        with open(DS/f"labels/{split}/{stem}.txt","w") as f:
            if box is not None:
                f.write("%d %.6f %.6f %.6f %.6f\n" % box)
    print(f"{split}: wrote {n} images")

build_split("train", N_TRAIN, 1)
build_split("val",   N_VAL,   2)

with open(DS/"data.yaml","w") as f:
    f.write(f"path: {DS.resolve()}\ntrain: images/train\nval: images/val\n"
            f"nc: 1\nnames: [hoarding]\n")
print("data.yaml written")


## 5 · Sanity-check a few samples (image + auto-label box)

In [ ]:
import matplotlib.pyplot as plt, glob, cv2
files=sorted(glob.glob(str(DS/"images/train/*.jpg")))[:6]
fig,axs=plt.subplots(2,3,figsize=(14,6))
for ax,fp in zip(axs.ravel(), files):
    img=cv2.cvtColor(cv2.imread(fp),cv2.COLOR_BGR2RGB); H,W=img.shape[:2]
    lab=fp.replace("images","labels").replace(".jpg",".txt")
    for line in open(lab):
        _,cx,cy,bw,bh=map(float,line.split())
        x1,y1=int((cx-bw/2)*W),int((cy-bh/2)*H); x2,y2=int((cx+bw/2)*W),int((cy+bh/2)*H)
        cv2.rectangle(img,(x1,y1),(x2,y2),(255,0,0),2)
    ax.imshow(img); ax.axis("off")
plt.tight_layout(); plt.show()


## 6 · Train YOLOv8n on the AMD GPU
~a few minutes on an Instinct GPU. `yolov8n.pt` (pretrained backbone) downloads from
Ultralytics on first run.

In [ ]:
from ultralytics import YOLO
model = YOLO("yolov8n.pt")            # start from COCO-pretrained weights
model.train(data=str(DS/"data.yaml"), epochs=EPOCHS, imgsz=IMG_W,
            batch=BATCH, device=DEVICE, project="runs", name="hoarding",
            patience=15, verbose=True)


## 7 · Evaluate + preview predictions on held-out synthetic frames

In [ ]:
import glob
metrics = model.val()
print("mAP50:", round(float(metrics.box.map50),3), "| mAP50-95:", round(float(metrics.box.map),3))

import matplotlib.pyplot as plt, cv2
val_imgs=sorted(glob.glob(str(DS/"images/val/*.jpg")))[:6]
fig,axs=plt.subplots(2,3,figsize=(14,6))
for ax,fp in zip(axs.ravel(), val_imgs):
    r=model(fp, verbose=False, conf=0.35)[0]
    im=cv2.cvtColor(r.plot(),cv2.COLOR_BGR2RGB)
    ax.imshow(im); ax.axis("off")
plt.tight_layout(); plt.show()


## 8 · Export the model for the main pipeline

In [ ]:
import shutil
best = "runs/hoarding/weights/best.pt"
shutil.copy(best, MODEL_OUT)
print("Exported ->", MODEL_OUT, "(copy this next to your main notebook)")


## 9 · OPTION B — add real frames to close the synthetic→real gap (recommended)
Synthetic-only training usually gets you a working demo, but a small batch of **real**
frames sharpens it a lot. Cheapest route uses your existing heuristic as an
auto-labeller, then you fix mistakes in Roboflow (free).

**Step 1 — pull frames from any cricket video** (your match file or a downloaded clip):

In [ ]:
# Extract 1 frame every 3 seconds into ./real_frames/
import os, subprocess
SRC_VIDEO = "./data/match.mp4"      # point at a real cricket video
os.makedirs("./real_frames", exist_ok=True)
if os.path.exists(SRC_VIDEO):
    subprocess.run(f'ffmpeg -y -i "{SRC_VIDEO}" -vf fps=1/3 ./real_frames/f_%04d.jpg',
                   shell=True)
    print("frames:", len(os.listdir("./real_frames")))
else:
    print("No video at", SRC_VIDEO, "- set SRC_VIDEO to a real cricket clip.")


**Step 2 — choose how to label them:**

- **Auto-label (fast start):** run the heuristic `detect_boundary_board` from the
  main notebook on each frame, write a YOLO `.txt` from the returned box, then *spot-fix*
  the wrong ones. Even 60–70% correct auto-labels save most of the work.
- **Manual / correction tool — Roboflow (free):** create a project at
  `https://roboflow.com`, upload `./real_frames`, draw/adjust one `hoarding` box per
  image, then export in **YOLOv8** format and drop it alongside the synthetic data.
- **Related public data for extra pretraining:** generic hoarding/billboard sets at
  `https://universe.roboflow.com/search?q=class:hoarding` and cricket field/ball sets at
  `https://universe.roboflow.com/search?q=class:cricket` (different domain, but useful
  for warming up the backbone).

**Step 3 — mix real + synthetic and re-run training (Cell 6).** Put real images in
`images/train` (or a separate folder added to `data.yaml`) and retrain. A 90/10
synthetic-to-real mix, or fine-tuning the synthetic model on ~150–300 real frames,
works well.

## 10 · Plug the trained model into the main pipeline
Copy `hoarding_best.pt` next to your main notebook, then **replace** the
`detect_surfaces` priority so YOLO is tried first and the heuristic stays as a
fallback. Paste this into the main notebook (Cell 13 area):

```python
from ultralytics import YOLO
import os, numpy as np
_HOARD_MODEL = None
def _load_hoard():
    global _HOARD_MODEL
    if _HOARD_MODEL is None and os.path.exists("hoarding_best.pt"):
        _HOARD_MODEL = YOLO("hoarding_best.pt")
    return _HOARD_MODEL

def detect_boundary_board_yolo(frame, conf=0.35):
    m = _load_hoard()
    if m is None:
        return None
    r = m(frame, verbose=False, conf=conf)[0]
    if len(r.boxes) == 0:
        return None
    b = r.boxes[int(r.boxes.conf.argmax())]
    x1, y1, x2, y2 = b.xyxy[0].cpu().numpy()
    quad = np.float32([[x1,y1],[x2,y1],[x2,y2],[x1,y2]])
    return ("hoarding", float(b.conf[0]), quad)
```

Then at the very top of `detect_surfaces`, before the heuristic candidates:

```python
    y = detect_boundary_board_yolo(frame)      # learned detector first
    if y is not None:
        cands.append(y)                        # conf as its score; usually wins auto
```

Because the YOLO box comes back as a 4-corner quad in the same format, the existing
warp + optical-flow tracker + ad-insertion code works unchanged — only the *source*
of the box changed from colour-rules to a trained model.
